<a href="https://colab.research.google.com/github/UdaraChamidu/Eye-Disease-Classification-With-Integrated-Chatbot/blob/main/clean_the_dataset_for_finetune_bioGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Repair the JSON File

In [24]:
pip install json_repair

In [26]:
!json_repair /content/train_dataset_1.json > /content/train_dataset_1_repaired.json

# Convert into jsonl format

In [27]:
import json

with open("/content/train_dataset_1_repaired.json", "r", encoding="utf-8") as f:
    data = json.load(f)

with open("/content/train_dataset_1.jsonl", "w", encoding="utf-8") as f:
    for item in data:
        json.dump({
            "prompt": item["prompt"].strip(),
            "completion": item["completion"].strip()
        }, f)
        f.write("\n")


# Fix issues of .jsonl data before finetune

In [28]:
import json

data = []
with open("/content/train_dataset_1.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        try:
            data.append(json.loads(line))
        except json.JSONDecodeError as e:
            print(f"Skipping invalid JSON line: {e}")


In [29]:
cleaned_data = []

for entry in data:
    prompt = entry.get("prompt", "").strip()
    completion = entry.get("completion", "").strip()

    # Check if prompt or completion is empty or suspiciously short
    if len(prompt) < 20 or len(completion) < 20:
        continue  # skip short or empty entries

    # Optional: Remove entries with unrelated topics (like cardiac symptoms)
    if "chest" in prompt.lower() or "throat" in prompt.lower():
        continue

    cleaned_data.append(entry)

print(f"Kept {len(cleaned_data)} entries out of {len(data)}")


Kept 16591 entries out of 16953


In [30]:
for entry in cleaned_data:
    entry["prompt"] = entry["prompt"].replace("\\n", "\n")
    entry["completion"] = entry["completion"].replace("\\n", "\n")


In [31]:
with open("cleaned_dataset.jsonl", "w", encoding="utf-8") as f:
    for entry in cleaned_data:
        json.dump(entry, f, ensure_ascii=False)
        f.write("\n")

print("✅ Cleaned dataset saved as cleaned_dataset.jsonl")


✅ Cleaned dataset saved as cleaned_dataset.jsonl


1. Prepare Your Environment
You'll need the following:

Python 3.8+

PyTorch

Hugging Face Transformers + Datasets

Optional: Use a GPU (e.g., Google Colab with T4 or A100)

In [21]:
pip install transformers datasets accelerate peft

2. Load and Inspect Your JSON Dataset


In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="biogpt_eyedisease_data.jsonl", split="train")
print(dataset[0])  # See one sample

3. Tokenize the Data

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("microsoft/BioGPT")

def tokenize_function(example):
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=512)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

4. Define the Training Setup

In [ ]:
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments

model = AutoModelForCausalLM.from_pretrained("microsoft/BioGPT")

training_args = TrainingArguments(
    output_dir="./biogpt_finetuned",
    per_device_train_batch_size=4,
    num_train_epochs=3,
    logging_dir="./logs",
    save_steps=500,
    logging_steps=100,
    evaluation_strategy="no",
    fp16=True,  # Use if on GPU
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
)

trainer.train()

5. Save the Fine-Tuned Model

In [ ]:
model.save_pretrained("finetuned_biogpt_eyedisease")
tokenizer.save_pretrained("finetuned_biogpt_eyedisease")

6. Try Inference

In [ ]:
from transformers import pipeline

qa = pipeline("text-generation", model="finetuned_biogpt_eyedisease", tokenizer=tokenizer)
result = qa("Question: What are the symptoms of glaucoma?\nAnswer:")
print(result[0]['generated_text'])